# LangGraph Module · Day 10 — Capstone: Invoice-Triage Agent

**Phase 1 · Module 1: Capstone** · ~2.5 hrs

Ten days of pieces — state, nodes, edges, conditional routing, human-in-the-loop, persistence — now snap together into **one agent** that does a real bank job: **triage an incoming invoice**.

```
  OCR (read the image)
       ↓
  CLASSIFY (what kind of invoice?)
       ↓
  VALIDATE (Pydantic — is it sane?)  ──✗──▶  FLAG for a human, stop
       ↓ ✓
  HUMAN-GATE (big amount? pause & ask)  ──reject──▶  stop
       ↓ approve / small
  SAVE to the database
```

### What I build today
- [ ] A **5-node pipeline**: OCR → classify → validate → human-gate → DB-write
- [ ] **Conditional routing**: invalid invoices skip the DB and go to a human
- [ ] A **human-gate** that *pauses* on big amounts using `interrupt()` + a checkpointer
- [ ] **Streaming** the run node-by-node with `graph.stream(...)`
- [ ] **Logging** every step, and **error recovery** so one bad OCR doesn't crash the run
- [ ] Uses today's Python lesson: the **validate** node is a Pydantic model

> **No API key needed** — every node is a plain Python function so you can watch the wiring. OCR and "classify" are mocked; the *graph structure* is the real, transferable skill.

## 0. Setup

Keyless. We import LangGraph's graph builder, a `MemorySaver` checkpointer (so the graph can **pause and resume** at the human-gate), and Pydantic for the validator node.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, field_validator, computed_field, ValidationError

print("invoice-triage capstone — keyless, runs offline")

invoice-triage capstone — keyless, runs offline


## 1. The state — one shared clipboard the whole pipeline writes on

Every node reads the state and returns a small dict that gets merged in. Think of it as a **case file** for one invoice that each desk adds a stamp to as it moves down the line.

In [2]:
class InvoiceState(TypedDict):
    image: str            # incoming file name (mock)
    text: str             # what OCR "read"
    fields: dict          # parsed vendor / amount / currency
    category: str         # classifier output
    valid: bool           # did validation pass?
    errors: list          # validation problems, if any
    status: str           # where we ended: SAVED / FLAGGED / REJECTED
    log: list             # a running trace of every step

def log(state, msg):
    "Append a line to the trace and print it — our lightweight logging."
    print("  •", msg)
    return state.get("log", []) + [msg]

☝ `log` is our tiny logging helper (Day 08's habit, kept simple): it prints the step **and** returns the growing trace so we can store it in state. In production this would be `structlog`; the pattern — *record what each node did* — is identical.

## 2. Node 1 — OCR (mock), with error recovery

A real OCR reads pixels; ours **looks up** canned text so the notebook stays keyless. The important part is the **error recovery**: if OCR can't read the file, the node doesn't crash — it records the problem in state and lets the graph route the case to a human.

In [3]:
# pretend "scanned images" -> raw text an OCR engine would return
FAKE_SCANS = {
    "inv_aws.png":    "INVOICE  Vendor: AWS  Amount: 420.50  Currency: GBP",
    "inv_big.png":    "INVOICE  Vendor: Oracle  Amount: 88000  Currency: GBP",
    "inv_bad.png":    "INVOICE  Vendor: Shady Ltd  Amount: -5  Currency: GBP",
    "inv_smudge.png": "@@@ unreadable @@@",            # OCR will fail to parse
}

def ocr_node(state: InvoiceState) -> dict:
    trace = log(state, f"OCR reading {state['image']}")
    text = FAKE_SCANS.get(state["image"], "")
    if not text or "unreadable" in text:
        # error recovery: don't crash — flag it
        return {"text": "", "status": "FLAGGED",
                "errors": ["OCR could not read the document"],
                "log": log({"log": trace}, "OCR FAILED -> will flag for human")}
    return {"text": text, "log": trace}

☝ Two exits from one node: clean text on success, or `status="FLAGGED"` + an error on failure — **no exception escapes**. That's error recovery in a graph: a failing node returns a *state update* describing the failure, and a later branch decides what to do. Nothing downstream ever sees a half-built case.

## 3. Node 2 — parse + classify

We split the OCR text into fields, then a simple rule-based **classifier** tags the invoice by vendor. (Swap this rule for an LLM later — the node's in/out contract never changes, which is the whole point of the graph.)

In [4]:
def parse_node(state: InvoiceState) -> dict:
    if state.get("status") == "FLAGGED":          # OCR already failed -> skip
        return {}
    trace = log(state, "parsing fields from text")
    parts = dict(p.split(": ", 1) for p in
                 [x.strip() for x in state["text"].split("  ") if ": " in x])
    fields = {
        "vendor":   parts.get("Vendor", "").strip(),
        "amount":   float(parts.get("Amount", 0)),
        "currency": parts.get("Currency", "").strip(),
    }
    return {"fields": fields, "log": trace}

def classify_node(state: InvoiceState) -> dict:
    if state.get("status") == "FLAGGED":
        return {}
    trace = log(state, "classifying invoice")
    vendor = state["fields"]["vendor"].lower()
    if any(k in vendor for k in ("aws", "oracle", "azure")):
        cat = "cloud-infra"
    elif "consult" in vendor:
        cat = "professional-services"
    else:
        cat = "general"
    return {"category": cat, "log": trace}

☝ Each node **guards** on `status == "FLAGGED"` and returns `{}` (no change) if the case already failed upstream — so a broken invoice glides past the remaining desks untouched. The classifier is a plain `if/elif`; the graph doesn't care whether that's rules or a model.

## 4. Node 3 — validate (today's Python lesson, as a node)

This is the Pydantic model from **Python Day 10**, dropped straight in as a node. It enforces the business rules (amount > 0, known currency) and derives `total_with_tax`. Pass → `valid=True`; fail → `valid=False` with the list of problems. The node **never raises** — it converts the `ValidationError` into state.

In [5]:
class ValidatedInvoice(BaseModel):
    vendor: str
    amount: float
    currency: str

    @field_validator("vendor")
    @classmethod
    def vendor_not_blank(cls, v):
        if not v.strip():
            raise ValueError("vendor is blank")
        return v.strip()

    @field_validator("amount")
    @classmethod
    def amount_positive(cls, v):
        if v <= 0:
            raise ValueError("amount must be > 0")
        return v

    @field_validator("currency")
    @classmethod
    def currency_known(cls, v):
        if v.upper() not in {"GBP", "USD", "EUR"}:
            raise ValueError(f"unknown currency {v!r}")
        return v.upper()

    @computed_field
    @property
    def total_with_tax(self) -> float:
        return round(self.amount * 1.20, 2)

def validate_node(state: InvoiceState) -> dict:
    if state.get("status") == "FLAGGED":
        return {}
    trace = log(state, "validating with Pydantic")
    try:
        inv = ValidatedInvoice(**state["fields"])
        f = {**state["fields"], "total_with_tax": inv.total_with_tax}
        return {"valid": True, "errors": [], "fields": f, "log": trace}
    except ValidationError as e:
        msgs = [x["msg"] for x in e.errors()]
        return {"valid": False, "errors": msgs,
                "log": log({"log": trace}, f"validation FAILED: {msgs}")}

☝ The exact model you wrote this morning is now the agent's quality gate. It hands back a **state update** (`valid`, `errors`) — the graph reads `valid` next to decide the invoice's fate. This is why we practised swallowing `ValidationError` into `.errors()`: a node returns data, never an exception.

## 5. The router — decide the invoice's fate

After validation we branch. This is a plain function returning the **name of the next node** — LangGraph's conditional-edge pattern from Day 05. Three outcomes:
- invalid (or OCR-flagged) → **human review** (never touch the DB)
- valid **and** large amount → **human-gate** (approval needed)
- valid and small → straight to **save**

In [6]:
BIG_AMOUNT = 10_000        # invoices over £10k need sign-off

def route_after_validate(state: InvoiceState) -> str:
    if state.get("status") == "FLAGGED" or not state.get("valid", False):
        return "flag"                      # bad data -> human review
    if state["fields"]["amount"] >= BIG_AMOUNT:
        return "human_gate"                # big -> ask a person
    return "save"                          # small & clean -> auto-save

☝ One function, three destinations — the graph calls it and jumps to whichever node name it returns. All the policy (*what needs a human?*) lives in this one readable place, separate from the work the nodes do. That separation is what keeps agents maintainable.

## 6. Human-gate, flag, and save nodes

- **`human_gate`** calls `interrupt()` — the graph **pauses** here and hands control back to you (the "approver"). When you resume with a decision, it continues. This needs a **checkpointer** so the paused state is saved.
- **`flag`** parks the case for a human to look at (bad data / OCR fail).
- **`save`** writes to our mock database.

In [7]:
DB = []          # mock "database" table

def human_gate_node(state: InvoiceState) -> dict:
    trace = log(state, f"PAUSING for approval — £{state['fields']['amount']:,.0f}")
    # interrupt() stops the graph and surfaces this payload to the caller
    decision = interrupt({
        "question": "Approve this large invoice?",
        "vendor": state["fields"]["vendor"],
        "amount": state["fields"]["amount"],
    })
    # ...execution resumes HERE once we send a decision back
    if decision == "approve":
        return {"log": log({"log": trace}, "human APPROVED")}
    return {"status": "REJECTED",
            "log": log({"log": trace}, "human REJECTED")}

def flag_node(state: InvoiceState) -> dict:
    return {"status": "FLAGGED",
            "log": log(state, f"FLAGGED for human review: {state.get('errors')}")}

def save_node(state: InvoiceState) -> dict:
    if state.get("status") == "REJECTED":      # human said no -> don't save
        return {"log": log(state, "not saved (rejected)")}
    DB.append(state["fields"])
    return {"status": "SAVED",
            "log": log(state, f"SAVED to DB (row {len(DB)})")}

☝ `interrupt(payload)` is human-in-the-loop from Day 06/07: it freezes the run, the checkpointer stores exactly where we stopped, and the payload (vendor + amount) is what a UI would show the approver. Execution **continues from the same line** when we resume — no re-running the earlier nodes.

## 7. Wire the graph

Nodes become boxes, edges become arrows. Note the **conditional edge** after `validate` (using the router from §5) and that `human_gate` flows into `save` — an approved big invoice still gets saved.

In [8]:
builder = StateGraph(InvoiceState)
builder.add_node("ocr", ocr_node)
builder.add_node("parse", parse_node)
builder.add_node("classify", classify_node)
builder.add_node("validate", validate_node)
builder.add_node("human_gate", human_gate_node)
builder.add_node("flag", flag_node)
builder.add_node("save", save_node)

builder.add_edge(START, "ocr")
builder.add_edge("ocr", "parse")
builder.add_edge("parse", "classify")
builder.add_edge("classify", "validate")
builder.add_conditional_edges("validate", route_after_validate,
                              {"human_gate": "human_gate", "flag": "flag", "save": "save"})
builder.add_edge("human_gate", "save")
builder.add_edge("flag", END)
builder.add_edge("save", END)

# checkpointer lets the graph pause at interrupt() and resume later
graph = builder.compile(checkpointer=MemorySaver())
print("graph compiled with 7 nodes")

graph compiled with 7 nodes


```mermaid
flowchart TD
    START --> ocr --> parse --> classify --> validate
    validate -->|valid & small| save
    validate -->|valid & big| human_gate --> save
    validate -->|invalid / OCR fail| flag --> END
    save --> END
```

## 8. Run A — a clean small invoice, **streamed**

`graph.stream(..., stream_mode="updates")` yields **one chunk per node** as it finishes, so you watch the case move down the line instead of waiting for the end. Every graph needs a `thread_id` (the checkpointer keys the case file on it).

In [9]:
cfg = {"configurable": {"thread_id": "inv-1"}}
start = {"image": "inv_aws.png", "fields": {}, "errors": [], "log": [], "status": ""}

print("STREAM (node-by-node):")
final = None
for chunk in graph.stream(start, cfg, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"[{node}] -> {({k: update[k] for k in update if k != 'log'})}")
    final = chunk

print("\nfinal status:", list(final.values())[0].get("status"))
print("rows in DB:", len(DB))

STREAM (node-by-node):
  • OCR reading inv_aws.png
[ocr] -> {'text': 'INVOICE  Vendor: AWS  Amount: 420.50  Currency: GBP'}
  • parsing fields from text
[parse] -> {'fields': {'vendor': 'AWS', 'amount': 420.5, 'currency': 'GBP'}}
  • classifying invoice
[classify] -> {'category': 'cloud-infra'}
  • validating with Pydantic
[validate] -> {'valid': True, 'errors': [], 'fields': {'vendor': 'AWS', 'amount': 420.5, 'currency': 'GBP', 'total_with_tax': 504.6}}
  • SAVED to DB (row 1)
[save] -> {'status': 'SAVED'}

final status: SAVED
rows in DB: 1


☝ Six nodes fire in order and the small clean invoice lands as **SAVED** — no human needed. Streaming gave us a live trace (great for a UI progress bar or debugging *which* node changed *what*). The `total_with_tax` your Pydantic model computed rode along in `fields`.

## 9. Run B — a **big** invoice: pause, ask, resume

`inv_big.png` is £88,000 — over the threshold. The stream will run up to `human_gate` and then **stop** at the interrupt. We inspect the pause, then **resume** with `Command(resume="approve")` and the graph continues from that exact spot into `save`.

In [10]:
cfg = {"configurable": {"thread_id": "inv-2"}}
start = {"image": "inv_big.png", "fields": {}, "errors": [], "log": [], "status": ""}

print("--- first run: expect a pause ---")
for chunk in graph.stream(start, cfg, stream_mode="updates"):
    if "__interrupt__" in chunk:
        payload = chunk["__interrupt__"][0].value
        print("PAUSED. Agent asks:", payload["question"],
              f"(£{payload['amount']:,.0f})")

# a human reviews and approves; resume from the interrupt
print("\n--- resuming with 'approve' ---")
for chunk in graph.stream(Command(resume="approve"), cfg, stream_mode="updates"):
    for node, update in chunk.items():
        print(f"[{node}] -> status={update.get('status')}")

print("\nrows in DB now:", len(DB))

--- first run: expect a pause ---
  • OCR reading inv_big.png
  • parsing fields from text
  • classifying invoice
  • validating with Pydantic
  • PAUSING for approval — £88,000
PAUSED. Agent asks: Approve this large invoice? (£88,000)

--- resuming with 'approve' ---
  • PAUSING for approval — £88,000
  • human APPROVED
[human_gate] -> status=None
  • SAVED to DB (row 2)
[save] -> status=SAVED

rows in DB now: 2


☝ The run **halted** at `human_gate`, surfaced the question, and waited. Sending `Command(resume="approve")` picked up **exactly where it stopped** — the earlier nodes did **not** re-run — and flowed into `save`. That's a durable approval workflow: the case can sit paused for minutes or days (the checkpointer holds it) until a person decides.

## 10. Run C — a **bad** invoice: routed away from the DB

`inv_bad.png` has `amount: -5`. Validation fails, the router sends it to `flag`, and it ends **FLAGGED** — the database is never touched. This is the whole reason for the validator gate: garbage never reaches the ledger.

In [11]:
cfg = {"configurable": {"thread_id": "inv-3"}}
start = {"image": "inv_bad.png", "fields": {}, "errors": [], "log": [], "status": ""}

result = graph.invoke(start, cfg)      # .invoke = run to the end, no streaming
print("\nfinal status:", result["status"])
print("errors:", result["errors"])
print("rows in DB (unchanged):", len(DB))

  • OCR reading inv_bad.png
  • parsing fields from text
  • classifying invoice
  • validating with Pydantic
  • validation FAILED: ['Value error, amount must be > 0']
  • FLAGGED for human review: ['Value error, amount must be > 0']

final status: FLAGGED
errors: ['Value error, amount must be > 0']
rows in DB (unchanged): 2


☝ Same pipeline, a different path: `validate` set `valid=False`, the router chose `flag`, and the invoice stopped safely with its error message intact. Notice we used `.invoke()` here (run straight to the end) instead of `.stream()` — same graph, you just pick how much you want to watch.

## 11. Recap — you shipped an agent

| Piece | Node(s) | Day it came from |
|-------|---------|------------------|
| shared **state** | `InvoiceState` TypedDict | Day 02 |
| **nodes & edges** | ocr → parse → classify → … | Day 03–04 |
| **conditional routing** | `route_after_validate` | Day 05 |
| **human-in-the-loop** | `interrupt()` + `Command(resume=...)` | Day 06–07 |
| **persistence / pause-resume** | `MemorySaver` checkpointer | Day 07 |
| **streaming** | `graph.stream(stream_mode="updates")` | today |
| **validation** | Pydantic `validate_node` | Python Day 10 |
| **error recovery** | flag-on-failure, `status` guards | today |

**Three exits, one graph:** clean invoices **SAVE**, big ones **pause for a human**, bad ones **FLAG** — and the DB only ever sees good data.

### Where a real LLM slots in (structure unchanged)
- `ocr_node` → a vision model or Textract reading a real PDF
- `classify_node` → an LLM tagging the invoice category
- `human_gate` → a Slack/Teams approval button hitting `Command(resume=...)`

The **graph** — state, routing, human-gate, validation, recovery — stays exactly as you built it. That is the capstone: the wiring is the product, the models are swappable parts.

> **No key needed** — the full triage flow above is keyless and offline.